# 02 · Fixtures & State

**Focus:** dependency injection for reusable setup and teardown.

A **fixture** is a function decorated with `@pytest.fixture` that produces something a test needs —
a database connection, a temp directory, a configured client. A test simply declares the fixture's
name as a **parameter**, and pytest injects the value. This is *dependency injection*: the test
asks for what it needs, pytest wires it up.

The magic keyword is **`yield`**. Everything *before* `yield` is setup; everything *after* is
teardown that runs once the test is done — even if the test fails. This is how you guarantee
cleanup (close connections, delete files) without `try/finally` in every test.

### Setup — point the shell at this course's tools
The `!` cells below run command-line tools (`pytest`, later `mutmut`, `playwright`). We prepend the
active kernel's `bin/` directory to `PATH` so those commands resolve to the versions installed for
**this course**, not some other Python on your machine. Run this cell first.

In [1]:
import sys, os
# The kernel's own interpreter lives in the course virtualenv's bin/ directory.
os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]
print("Using Python:", sys.executable)
!pytest --version

Using Python: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python


pytest 9.1.1


### A fake database connection
We simulate a DB connection that must be opened and *closed*.

In [2]:
%%writefile nb02_db.py
class FakeDBConnection:
    def __init__(self):
        self.open = False
        self.rows = []

    def connect(self):
        self.open = True
        print("  [db] connection OPENED")

    def close(self):
        self.open = False
        print("  [db] connection CLOSED")

    def insert(self, row):
        if not self.open:
            raise RuntimeError("connection is closed")
        self.rows.append(row)

    def count(self):
        return len(self.rows)

Writing nb02_db.py


### A fixture with `yield`
`setup → yield value → teardown`. The `print` lines let us *see* the ordering when we pass `-s`.

In [3]:
%%writefile test_nb02.py
import pytest
from nb02_db import FakeDBConnection

@pytest.fixture
def db():
    # ---- setup ----
    conn = FakeDBConnection()
    conn.connect()
    yield conn            # <-- the test runs here, receiving `conn`
    # ---- teardown ----
    conn.close()

def test_insert_one(db):
    print("  [test] running test_insert_one")
    db.insert({"id": 1})
    assert db.count() == 1

def test_insert_many(db):
    print("  [test] running test_insert_many")
    db.insert({"id": 1})
    db.insert({"id": 2})
    assert db.count() == 2
    # NOTE: each test gets a FRESH db — count starts at 0 again.

Writing test_nb02.py


### Watch the setup/teardown order
`-s` disables output capturing so we can see the `print`s. Notice the connection is **opened before
each test and closed after each test** — the fixture runs once per test by default (`scope="function"`).

In [4]:
!pytest -v -s test_nb02.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 2 items                                                              

test_nb02.py::test_insert_one   [db] connection OPENED
  [test] running test_insert_one
PASSED  [db] connection CLOSED

test_nb02.py::test_insert_many   [db] connection OPENED
  [test] running test_insert_many
PASSED  [db] connection CLOSED


============================== 2 passed in 0.01s =====

Read the ordering: `OPENED → test runs → CLOSED`, twice. Each test received its **own** freshly
connected `db`, and teardown closed it even though we never wrote a single `close()` call in a test.
That isolation — no shared mutable state leaking between tests — is the whole point of fixtures.

### ⚠️ Common pitfall — fixture scope leaks state
The default `scope="function"` gives each test a fresh fixture. Widen it to `scope="module"` or
`"session"` to share expensive setup — but then **state persists between tests**, and a mutation in
one test can leak into the next, creating order-dependent flakiness. Share scope only for *immutable*
or *safely-resettable* resources.

### 🔬 Try it yourself
Here the same `db` is shared across two tests via `scope="module"`. **Predict:** in `test_b`, is the
row count `1` or `2`? Run it — the passing `== 2` proves `test_a`'s insert leaked in. Change the scope
back to `"function"` and re-run to watch the leak disappear (and `test_b`'s assert then fail).

In [5]:
%%writefile test_nb02_tryit.py
import pytest
from nb02_db import FakeDBConnection

@pytest.fixture(scope="module")          # <-- shared across every test in this file
def shared_db():
    conn = FakeDBConnection(); conn.connect()
    yield conn
    conn.close()

def test_a(shared_db):
    shared_db.insert({"id": 1})
    assert shared_db.count() == 1

def test_b(shared_db):
    shared_db.insert({"id": 2})
    assert shared_db.count() == 2        # only true because state LEAKED from test_a

Writing test_nb02_tryit.py


In [6]:
!pytest -v -s test_nb02_tryit.py

============================= test session starts ==============================
platform darwin -- Python 3.12.11, pytest-9.1.1, pluggy-1.6.0 -- /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/ajinkyak/Desktop/JULY-WORK/a - training_assignments/explorable_codelesson
plugins: syrupy-5.5.3, mock-3.15.1, cov-7.1.0, xdist-3.8.0, asyncio-1.4.0, time-machine-3.2.0, anyio-4.14.2, base-url-2.1.0, playwright-0.8.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 2 items                                                              

test_nb02_tryit.py::test_a   [db] connection OPENED
PASSED
test_nb02_tryit.py::test_b 

PASSED  [db] connection CLOSED


============================== 2 passed in 0.02s ===============================


### What you learned
- `@pytest.fixture` + a parameter of the same name = dependency injection.
- Code before `yield` is setup; code after `yield` is guaranteed teardown.
- By default a fixture re-runs for every test, giving each test a clean slate.

Next up: **parametrization** — running one test against a whole matrix of inputs.